# 🌫️ AQI Forecaster — Data Science Notebook
## Exploratory Data Analysis + Model Training + Evaluation

**Author:** Mehak Khan  
**Tech Stack:** Python · Pandas · Scikit-learn · RandomForest · GradientBoosting · Matplotlib · Seaborn

---
### Notebook Structure
1. Data Generation & Loading
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Model Training (RF + GB Ensemble)
5. Model Evaluation & Metrics
6. 7-Day Forecast Visualization
7. Feature Importance
8. City Comparison

In [ ]:
# ── Imports ──────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

from backend.utils.data_generator import (
    generate_historical_aqi, generate_historical_weather,
    get_aqi_category, CITY_PROFILES
)
from backend.ml.forecaster import make_features, FEATURE_COLS, TARGET_COL

# Plot style
plt.style.use('dark_background')
sns.set_palette('husl')
COLORS = ['#3b82f6','#22c55e','#f59e0b','#ef4444','#8b5cf6','#06b6d4']
AQI_COLORS = {'Good':'#00e400','Satisfactory':'#ffff00','Moderate':'#ff7e00',
               'Poor':'#ff0000','Very Poor':'#8f3f97','Severe':'#7e0023'}

print('✅ Imports successful')

## 1. Data Generation & Loading

In [ ]:
# Generate 365 days of data for Delhi (primary analysis city)
CITY = 'Delhi'
DAYS = 365

aqi_df     = generate_historical_aqi(CITY, days=DAYS)
weather_df = generate_historical_weather(CITY, days=DAYS)

print(f'AQI records:     {len(aqi_df):,}')
print(f'Weather records: {len(weather_df):,}')
print(f'Date range:      {aqi_df.recorded_at.min().date()} → {aqi_df.recorded_at.max().date()}')
aqi_df.head()

In [ ]:
print('AQI Statistics:')
print(aqi_df[['aqi','pm25','pm10','no2','so2','co','o3']].describe().round(2))

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f'AQI EDA — {CITY} ({DAYS} Days)', fontsize=16, fontweight='bold', color='white')

# 1. Daily AQI time series
daily = aqi_df.set_index('recorded_at').resample('D')['aqi'].mean()
ax = axes[0,0]
ax.fill_between(daily.index, daily.values, alpha=0.3, color='#3b82f6')
ax.plot(daily.index, daily.values, color='#3b82f6', linewidth=1.5)
for lo, hi, name, color in [(0,50,'Good','#00e400'),(51,100,'Sat','#ffff00'),
    (101,200,'Moderate','#ff7e00'),(201,300,'Poor','#ff0000'),
    (301,400,'V.Poor','#8f3f97'),(401,500,'Severe','#7e0023')]:
    ax.axhspan(lo, hi, alpha=0.06, color=color)
ax.set_title('Daily Mean AQI (365 days)', color='white')
ax.set_ylabel('AQI'); ax.set_xlabel('')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

# 2. AQI distribution
ax = axes[0,1]
ax.hist(aqi_df['aqi'], bins=50, color='#3b82f6', edgecolor='#111', alpha=0.8)
ax.axvline(aqi_df['aqi'].mean(), color='#f59e0b', linestyle='--', label=f'Mean: {aqi_df["aqi"].mean():.0f}')
ax.axvline(aqi_df['aqi'].median(), color='#22c55e', linestyle='--', label=f'Median: {aqi_df["aqi"].median():.0f}')
ax.set_title('AQI Distribution', color='white')
ax.set_xlabel('AQI'); ax.set_ylabel('Frequency')
ax.legend()

# 3. Monthly boxplot
ax = axes[1,0]
aqi_df['month'] = aqi_df['recorded_at'].dt.month
monthly_data = [aqi_df[aqi_df['month']==m]['aqi'].values for m in range(1,13)]
bp = ax.boxplot(monthly_data, patch_artist=True,
                boxprops=dict(facecolor='#1e3a5f', color='#3b82f6'),
                medianprops=dict(color='#f59e0b', linewidth=2),
                whiskerprops=dict(color='#5a6380'),
                capprops=dict(color='#5a6380'),
                flierprops=dict(marker='.', color='#5a6380', alpha=0.3))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_title('Monthly AQI Distribution', color='white')
ax.set_ylabel('AQI')

# 4. Hourly pattern
ax = axes[1,1]
aqi_df['hour'] = aqi_df['recorded_at'].dt.hour
hourly = aqi_df.groupby('hour')['aqi'].mean()
ax.plot(hourly.index, hourly.values, color='#22c55e', linewidth=2, marker='o', markersize=4)
ax.fill_between(hourly.index, hourly.values, alpha=0.2, color='#22c55e')
ax.set_title('Average AQI by Hour of Day', color='white')
ax.set_xlabel('Hour'); ax.set_ylabel('Mean AQI')
ax.set_xticks(range(0,24,2))

plt.tight_layout()
plt.savefig('../data/eda_overview.png', dpi=150, bbox_inches='tight', facecolor='#0a0d14')
plt.show()
print('Plot saved → data/eda_overview.png')

In [ ]:
# Correlation heatmap: AQI vs weather
merged = pd.merge(
    aqi_df[['recorded_at','aqi','pm25','pm10','no2','so2','co','o3']],
    weather_df[['recorded_at','temperature','humidity','wind_speed','pressure','visibility','rainfall']],
    on='recorded_at'
)

fig, ax = plt.subplots(figsize=(12, 8))
corr = merged.drop(columns='recorded_at').corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 9})
ax.set_title(f'Feature Correlation Matrix — {CITY}', fontsize=14, color='white', pad=15)
plt.tight_layout()
plt.savefig('../data/correlation_matrix.png', dpi=150, bbox_inches='tight', facecolor='#0a0d14')
plt.show()

In [ ]:
# AQI Category pie
cat_counts = aqi_df['category'].value_counts()
fig, ax = plt.subplots(figsize=(7, 7))
colors = [AQI_COLORS.get(c, '#888') for c in cat_counts.index]
wedges, texts, autotexts = ax.pie(
    cat_counts.values, labels=cat_counts.index,
    colors=colors, autopct='%1.1f%%',
    startangle=90, pctdistance=0.8,
    wedgeprops=dict(edgecolor='#0a0d14', linewidth=2)
)
for t in texts: t.set_color('white'); t.set_fontsize(11)
for at in autotexts: at.set_color('black'); at.set_fontsize(9); at.set_fontweight('bold')
ax.set_title(f'AQI Category Distribution\n{CITY} — {DAYS} Days', color='white', fontsize=14)
plt.savefig('../data/category_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0a0d14')
plt.show()

## 3. Feature Engineering

In [ ]:
df = make_features(aqi_df, weather_df)
print(f'Feature matrix shape: {df.shape}')
print(f'Features used: {[c for c in FEATURE_COLS if c in df.columns]}')
df[['date','aqi_mean','aqi_lag_1','aqi_roll_mean_7','temp_mean','humidity_mean']].tail(10)

## 4. Model Training

In [ ]:
available_features = [c for c in FEATURE_COLS if c in df.columns]
X = df[available_features].values
y = df[TARGET_COL].values

tscv = TimeSeriesSplit(n_splits=3)
splits = list(tscv.split(X))
train_idx, test_idx = splits[-1]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
dates_test = df['date'].values[test_idx]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train samples: {len(X_train)}')
print(f'Test  samples: {len(X_test)}')
print(f'Features:      {X_train.shape[1]}')

In [ ]:
print('Training Random Forest...')
rf = RandomForestRegressor(n_estimators=200, max_depth=12, min_samples_leaf=3, n_jobs=-1, random_state=42)
rf.fit(X_train_s, y_train)
preds_rf = rf.predict(X_test_s)

print('Training Gradient Boosting...')
gb = GradientBoostingRegressor(n_estimators=150, max_depth=5, learning_rate=0.08, subsample=0.8, random_state=42)
gb.fit(X_train_s, y_train)
preds_gb = gb.predict(X_test_s)

# Ensemble
preds_ens = 0.55 * preds_rf + 0.45 * preds_gb
print('✅ Models trained!')

## 5. Model Evaluation

In [ ]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    print(f'{name:<22} MAE={mae:6.2f}  RMSE={rmse:6.2f}  R²={r2:.4f}  MAPE={mape:.2f}%')
    return dict(model=name, mae=round(mae,2), rmse=round(rmse,2), r2=round(r2,4), mape=round(mape,2))

print('='*75)
print(f'{'Model':<22} {'MAE':>8}  {'RMSE':>8}  {'R²':>8}  {'MAPE':>8}')
print('='*75)
m1 = evaluate('Random Forest',        y_test, preds_rf)
m2 = evaluate('Gradient Boosting',    y_test, preds_gb)
m3 = evaluate('RF+GB Ensemble (55/45)', y_test, preds_ens)
print('='*75)

metrics_df = pd.DataFrame([m1, m2, m3])
metrics_df

In [ ]:
# Prediction vs Actual plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Model Evaluation — {CITY}', fontsize=14, color='white')

ax = axes[0]
ax.plot(dates_test[:60], y_test[:60], label='Actual', color='#3b82f6', linewidth=2)
ax.plot(dates_test[:60], preds_ens[:60], label='Ensemble Pred', color='#f59e0b', linewidth=2, linestyle='--')
ax.fill_between(dates_test[:60],
    preds_ens[:60] - 20, preds_ens[:60] + 20,
    alpha=0.15, color='#f59e0b', label='±20 AQI band')
ax.set_title('Predicted vs Actual AQI (first 60 test days)')
ax.set_ylabel('AQI'); ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

# Scatter
ax = axes[1]
ax.scatter(y_test, preds_ens, alpha=0.4, s=20,
           c=[int(v) for v in preds_ens], cmap='RdYlGn_r', vmin=0, vmax=400)
mn, mx = y_test.min(), y_test.max()
ax.plot([mn, mx], [mn, mx], 'w--', linewidth=1.5, label='Perfect')
ax.set_title(f'Scatter: Actual vs Predicted  R²={m3["r2"]}')
ax.set_xlabel('Actual AQI'); ax.set_ylabel('Predicted AQI')
ax.legend()

plt.tight_layout()
plt.savefig('../data/model_evaluation.png', dpi=150, bbox_inches='tight', facecolor='#0a0d14')
plt.show()

In [ ]:
# Model comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Model Comparison', fontsize=13, color='white')
model_names = ['Random Forest', 'Grad. Boosting', 'Ensemble']

for ax, metric, vals, better in zip(
    axes,
    ['MAE (lower=better)', 'RMSE (lower=better)', 'R² (higher=better)'],
    [[m1['mae'], m2['mae'], m3['mae']],
     [m1['rmse'], m2['rmse'], m3['rmse']],
     [m1['r2'],  m2['r2'],  m3['r2']]],
    ['min', 'min', 'max']
):
    bars = ax.bar(model_names, vals, color=['#3b82f6','#22c55e','#f59e0b'], edgecolor='#0a0d14')
    best_idx = np.argmin(vals) if better == 'min' else np.argmax(vals)
    bars[best_idx].set_edgecolor('white'); bars[best_idx].set_linewidth(2)
    ax.set_title(metric, color='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', color='white', fontsize=10)
    ax.set_xticklabels(model_names, fontsize=8)

plt.tight_layout()
plt.savefig('../data/model_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0a0d14')
plt.show()

## 6. 7-Day Forecast Visualization

In [ ]:
from backend.ml.forecaster import train_city_model, forecast_city
import joblib
from pathlib import Path

# Train & save model
metrics = train_city_model(CITY, df)
print('Training metrics:', {k:v for k,v in metrics.items() if k not in ['model_path','feature_importance']})

# Generate 7-day forecast
forecasts = forecast_city(CITY, horizon=7)
fc_df = pd.DataFrame(forecasts)
fc_df

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Last 14 days actual
hist14 = df.tail(14)
ax.plot(hist14['date'], hist14['aqi_mean'], 'o-', color='#3b82f6', linewidth=2, markersize=5, label='Historical AQI')

# Forecast
fc_dates  = pd.to_datetime(fc_df['date'])
fc_values = fc_df['predicted_aqi'].values
fc_lower  = fc_df['confidence_lower'].values
fc_upper  = fc_df['confidence_upper'].values
fc_colors = fc_df['color'].values

# Connect last hist point to first forecast
ax.plot([hist14['date'].iloc[-1], fc_dates[0]],
        [hist14['aqi_mean'].iloc[-1], fc_values[0]],
        '--', color='#5a6380', linewidth=1.5)

ax.plot(fc_dates, fc_values, 'o--', color='#f59e0b', linewidth=2, markersize=7, label='Forecast', zorder=5)
ax.fill_between(fc_dates, fc_lower, fc_upper, alpha=0.2, color='#f59e0b', label='95% CI')

# Color each forecast dot by AQI category
for date, val, color in zip(fc_dates, fc_values, fc_colors):
    ax.scatter(date, val, color=color, s=80, zorder=10, edgecolors='white', linewidths=1)
    ax.annotate(str(val), (date, val), textcoords='offset points',
                xytext=(0, 12), ha='center', fontsize=9, color='white', fontweight='bold')

# AQI bands
for lo, hi, name, color in [(0,50,'Good','#00e400'),(51,100,'Sat.','#ffff00'),
    (101,200,'Moderate','#ff7e00'),(201,300,'Poor','#ff0000'),
    (301,400,'V.Poor','#8f3f97')]:
    ax.axhspan(lo, hi, alpha=0.04, color=color)

ax.axvline(pd.Timestamp.now(), color='#5a6380', linestyle=':', linewidth=1.5, label='Today')
ax.set_title(f'7-Day AQI Forecast — {CITY}  (RF+GB Ensemble)', fontsize=14, color='white')
ax.set_ylabel('AQI'); ax.legend(loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)
plt.tight_layout()
plt.savefig('../data/forecast_7day.png', dpi=150, bbox_inches='tight', facecolor='#0a0d14')
plt.show()
print('Forecast saved → data/forecast_7day.png')

## 7. Feature Importance

In [ ]:
fi = pd.Series(rf.feature_importances_, index=available_features).sort_values(ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors_fi = ['#3b82f6' if 'aqi' in name or 'lag' in name or 'roll' in name
              else '#22c55e' if any(w in name for w in ['temp','humid','wind','pressure'])
              else '#f59e0b'
              for name in fi.index]
bars = ax.barh(fi.index, fi.values, color=colors_fi, edgecolor='#0a0d14')
ax.set_title('Top 20 Feature Importances (Random Forest)', fontsize=13, color='white')
ax.set_xlabel('Importance')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3b82f6', label='AQI / Lag features'),
    Patch(facecolor='#22c55e', label='Weather features'),
    Patch(facecolor='#f59e0b', label='Time / Calendar features'),
]
ax.legend(handles=legend_elements, loc='lower right')

for bar, val in zip(bars, fi.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8, color='#9aa3b8')

plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#0a0d14')
plt.show()

## 8. Multi-City AQI Comparison

In [ ]:
COMPARE_CITIES = ['Delhi','Mumbai','Bangalore','Chennai','Kolkata','Hyderabad','Jaipur','Lucknow']

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle('Monthly AQI Comparison — Major Indian Cities', fontsize=15, color='white')
axes = axes.flatten()

for ax, city in zip(axes, COMPARE_CITIES):
    try:
        city_aqi = generate_historical_aqi(city, days=365)
        city_daily = city_aqi.set_index('recorded_at').resample('D')['aqi'].mean()
        
        # Color by AQI level
        ax.fill_between(city_daily.index, city_daily.values, alpha=0.3, color='#3b82f6')
        ax.plot(city_daily.index, city_daily.values, linewidth=1, color='#3b82f6')
        
        mean_aqi = int(city_aqi['aqi'].mean())
        cat_color = ('#ff0000' if mean_aqi > 200 else '#ff7e00' if mean_aqi > 100
                     else '#ffff00' if mean_aqi > 50 else '#00e400')
        
        ax.set_title(f'{city}  (avg: {mean_aqi})', color=cat_color, fontsize=10)
        ax.set_ylim(0, 500)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
        ax.tick_params(labelsize=7)
    except Exception as e:
        ax.set_title(f'{city} (error)', color='red')

plt.tight_layout()
plt.savefig('../data/city_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0a0d14')
plt.show()
print('\n✅ All plots saved to data/ directory')

In [ ]:
# ── Train all 20 city models ──────────────────────────
from backend.utils.data_generator import generate_all_cities

all_metrics = []
for city_name, aqi_data, wx_data in generate_all_cities(days=365):
    try:
        feat_df = make_features(aqi_data, wx_data)
        m = train_city_model(city_name, feat_df)
        all_metrics.append(m)
        print(f'  ✅ {city_name:<20} MAE={m["mae"]:6.2f}  R²={m["r2"]:.3f}')
    except Exception as e:
        print(f'  ❌ {city_name}: {e}')

print(f'\n✅ Trained {len(all_metrics)}/20 city models successfully')

# Summary table
summary_df = pd.DataFrame(all_metrics)[['city','mae','rmse','r2','mape','training_samples']]
summary_df.sort_values('mae').round(3)